In [3]:
# ============================================================
# Task 4: Training Data Preparation and Fine-tuning YOLOv8
# COMP6001 Assignment 1
# Uses COCOBlur + NAFNet deblurring + YOLOv8 fine-tuning
# ============================================================

import subprocess
!pip install ultralytics pycocotools

import gc, os, json, shutil, time, warnings  # ← added gc
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from pycocotools.coco import COCO
from ultralytics import YOLO

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ── 1. Paths ─────────────────────────────────────────────────
# Add these datasets to your Kaggle notebook:
# 1. iamravigarg/cocoblur       → blurry COCO val images
# 2. COCO 2017 val images+labels → sharp images + annotations

# ── 1. Updated Paths for Task 4 ────────────────────────────────

# Directory containing the blurry COCO validation images
COCO_BLUR_DIR   = Path("/kaggle/input/datasets/iamravigarg/cocoblur/val2017_blurred_deterministic")

# Directory containing the original sharp COCO validation images
COCO_SHARP_DIR  = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017")

# Path to the official COCO 2017 validation instances (annotations)
COCO_ANNOT_FILE = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_val2017.json")

# Path for the NAFNet weights (local directory where weights will be downloaded)
NAFNET_WEIGHTS  = Path("./weights/NAFNet-GoPro-width64.pth")

# Output directory for training logs and the processed dataset
OUTPUT_DIR      = Path("/kaggle/working/outputs/task4")
DATASET_DIR     = OUTPUT_DIR / "dataset"

for d in [OUTPUT_DIR, DATASET_DIR,
          DATASET_DIR/"images/train", DATASET_DIR/"images/val",
          DATASET_DIR/"labels/train", DATASET_DIR/"labels/val"]:
    d.mkdir(parents=True, exist_ok=True)

# How many images to use (increase for better training)
MAX_TRAIN = 300
MAX_VAL   = 50
EPOCHS    = 20
IMG_SIZE  = 640
BATCH     = 8


# ── 2. NAFNet (reuse from Task 2) ────────────────────────────

class LayerNorm2d(nn.LayerNorm):
    def forward(self, x):
        return super().forward(x.permute(0,2,3,1)).permute(0,3,1,2)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.conv1 = nn.Conv2d(c, c*2, 1)
        self.conv2 = nn.Conv2d(c*2, c*2, 3, 1, 1, groups=c*2)
        self.conv3 = nn.Conv2d(c, c, 1)
        self.sca   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1))
        self.sg    = SimpleGate()
        self.conv4 = nn.Conv2d(c, c*2, 1)
        self.conv5 = nn.Conv2d(c, c, 1)
        self.beta  = nn.Parameter(torch.ones(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.ones(1, c, 1, 1))

    def forward(self, inp):
        x = self.sg(self.conv2(self.conv1(self.norm1(inp))))
        y = inp + self.conv3(x * self.sca(x)) * self.beta
        return y + self.conv5(self.sg(self.conv4(self.norm2(y)))) * self.gamma

class NAFNet(nn.Module):
    def __init__(self, width=64, enc_blks=[2,2,4,8],
                 middle_blk_num=12, dec_blks=[2,2,2,2]):
        super().__init__()
        self.intro       = nn.Conv2d(3, width, 3, 1, 1)
        self.ending      = nn.Conv2d(width, 3, 3, 1, 1)
        self.encoders    = nn.ModuleList()
        self.downs       = nn.ModuleList()
        self.middle_blks = nn.ModuleList()
        self.ups         = nn.ModuleList()
        self.decoders    = nn.ModuleList()
        chan = width
        for n in enc_blks:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(n)]))
            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for n in dec_blks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(n)]))
        self.padder_size = 2 ** len(self.encoders)

    def forward(self, inp):
            B, C, H, W = inp.shape
            # Calculate padding needed to make H and W multiples of padder_size
            ph = (self.padder_size - H % self.padder_size) % self.padder_size
            pw = (self.padder_size - W % self.padder_size) % self.padder_size
            
            # 1. Pad the input
            x = F.pad(inp, (0, pw, 0, ph), mode='reflect')
            
            # 2. Pass through the network
            x = self.intro(x)
            encs = []
            for enc, down in zip(self.encoders, self.downs):
                x = enc(x)
                encs.append(x)
                x = down(x)
            
            x = self.middle_blks(x)
            
            for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):
                x = dec(up(x) + skip)
                
            x = self.ending(x)
    
            # 3. FIX: Slice the padded output back to (H, W) BEFORE adding it to 'inp'
            # This ensures the global skip connection has matching tensor shapes
            return x[:, :, :H, :W] + inp

def load_nafnet():
    NAFNET_WEIGHTS.parent.mkdir(exist_ok=True)
    if NAFNET_WEIGHTS.exists() and NAFNET_WEIGHTS.stat().st_size < 10*1024*1024:
        NAFNET_WEIGHTS.unlink()
    if not NAFNET_WEIGHTS.exists():
        print("Downloading NAFNet weights from Google Drive (~140 MB)…")
        import gdown
        subprocess.run(["pip", "install", "gdown", "-q"], check=True)
        gdown.download(id="14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X",
                       output=str(NAFNET_WEIGHTS), quiet=False)
    ckpt  = torch.load(NAFNET_WEIGHTS, map_location=device)
    state = ckpt.get("params_ema") or ckpt.get("params") or ckpt
    width      = state["intro.weight"].shape[0]
    num_stages = len({k.split(".")[1] for k in state if k.startswith("downs.")})
    def nblocks(prefix):
        idxs = {int(k[len(prefix):].split(".")[0]) for k in state
                if k.startswith(prefix) and k[len(prefix):].split(".")[0].isdigit()}
        return max(idxs) + 1 if idxs else 0
    enc_blks       = [nblocks(f"encoders.{i}.") for i in range(num_stages)]
    dec_blks       = [nblocks(f"decoders.{i}.") for i in range(num_stages)]
    middle_blk_num = nblocks("middle_blks.")
    model = NAFNet(width=width, enc_blks=enc_blks,
                   middle_blk_num=middle_blk_num, dec_blks=dec_blks)
    model_keys = set(model.state_dict().keys())
    model.load_state_dict({k: v for k, v in state.items() if k in model_keys}, strict=False)
    return model.eval().to(device)


@torch.inference_mode()
def deblur_image(model, img_bgr):
    """Deblur a BGR uint8 image. Returns BGR uint8."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    t = torch.from_numpy(img_rgb.transpose(2,0,1)).unsqueeze(0).to(device)
    out = model(t).clamp(0,1)[0].cpu().numpy().transpose(1,2,0)
    torch.cuda.empty_cache()  # ← added
    return cv2.cvtColor((out*255).astype(np.uint8), cv2.COLOR_RGB2BGR)


# ── 3. COCO → YOLO label conversion ──────────────────────────

# COCO has 80 classes — map category_id to 0-indexed YOLO class
def build_coco_yolo_maps(coco):
    """Returns (cat_id_to_yolo_idx, yolo_idx_to_name)."""
    cats = coco.loadCats(coco.getCatIds())
    cats_sorted = sorted(cats, key=lambda c: c["id"])
    cat_to_idx  = {c["id"]: i for i, c in enumerate(cats_sorted)}
    idx_to_name = {i: c["name"] for i, c in enumerate(cats_sorted)}
    return cat_to_idx, idx_to_name


def coco_ann_to_yolo(ann, img_w, img_h, cat_to_idx):
    """Convert one COCO annotation to YOLO format string."""
    x, y, w, h = ann["bbox"]          # COCO: top-left x,y + width,height
    cx = (x + w/2) / img_w
    cy = (y + h/2) / img_h
    nw = w / img_w
    nh = h / img_h
    # Clamp to [0,1]
    cx, cy, nw, nh = [max(0.0, min(1.0, v)) for v in [cx, cy, nw, nh]]
    cls = cat_to_idx[ann["category_id"]]
    return f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"


def write_yolo_label(label_path, lines):
    with open(label_path, "w") as f:
        f.write("\n".join(lines))


# ── 4. Build dataset ──────────────────────────────────────────

def build_dataset(coco, cat_to_idx, nafnet_model,
                  blur_dir, sharp_dir,
                  max_train=MAX_TRAIN, max_val=MAX_VAL):
    """
    Train split : deblurred COCOBlur images + COCO annotations
    Val   split : original sharp COCO images + COCO annotations
    """
    all_img_ids = coco.getImgIds()
    print(f"Total COCO val images: {len(all_img_ids)}")

    # Filter to images that exist in the blur directory
    IMG_EXTS = {".jpg", ".jpeg", ".png"}
    blur_files = {p.stem: p for p in blur_dir.rglob("*")
                  if p.suffix.lower() in IMG_EXTS}

    valid_ids = []
    for img_id in all_img_ids:
        info = coco.loadImgs(img_id)[0]
        stem = Path(info["file_name"]).stem
        if stem in blur_files and coco.getAnnIds(imgIds=img_id):
            valid_ids.append(img_id)

    print(f"Matched {len(valid_ids)} images with blur + annotations")

    # Stratify by annotation density (more annotations = more complex scene)
    # Use first max_train+max_val images, sorted by annotation count
    ann_counts = [(img_id, len(coco.getAnnIds(imgIds=img_id)))
                  for img_id in valid_ids]
    ann_counts.sort(key=lambda x: x[1], reverse=True)

    selected   = ann_counts[:max_train + max_val]
    train_ids  = [x[0] for x in selected[:max_train]]
    val_ids    = [x[0] for x in selected[max_train:max_train + max_val]]

    print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")

    train_stats = {"deblurred": 0, "labels": 0, "skipped": 0}
    val_stats   = {"copied": 0,   "labels": 0, "skipped": 0}

    # ── Train: deblur blur images, use COCO labels ────────────
    print("\nPreparing training set (deblurring COCOBlur images)…")
    for img_id in tqdm(train_ids, desc="Train"):
        info  = coco.loadImgs(img_id)[0]
        stem  = Path(info["file_name"]).stem
        ext   = Path(info["file_name"]).suffix or ".jpg"

        blur_path = blur_files.get(stem)
        if blur_path is None:
            train_stats["skipped"] += 1; continue

        img_bgr = cv2.imread(str(blur_path))
        if img_bgr is None:
            train_stats["skipped"] += 1; continue

        # Deblur
        deb_bgr = deblur_image(nafnet_model, img_bgr)
        out_img = DATASET_DIR / "images" / "train" / f"{stem}.jpg"
        cv2.imwrite(str(out_img), deb_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        train_stats["deblurred"] += 1

        # COCO annotations → YOLO labels
        ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
        anns    = coco.loadAnns(ann_ids)
        h, w    = img_bgr.shape[:2]
        lines   = [coco_ann_to_yolo(a, w, h, cat_to_idx)
                   for a in anns if a["bbox"][2] > 2 and a["bbox"][3] > 2]
        if lines:
            write_yolo_label(DATASET_DIR/"labels"/"train"/f"{stem}.txt", lines)
            train_stats["labels"] += 1

    # ── Val: use sharp images + COCO labels ───────────────────
    print("\nPreparing validation set (sharp COCO images)…")
    for img_id in tqdm(val_ids, desc="Val"):
        info       = coco.loadImgs(img_id)[0]
        sharp_path = sharp_dir / info["file_name"]
        stem       = Path(info["file_name"]).stem

        if not sharp_path.exists():
            val_stats["skipped"] += 1; continue

        shutil.copy(sharp_path, DATASET_DIR/"images"/"val"/f"{stem}.jpg")
        val_stats["copied"] += 1

        ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
        anns    = coco.loadAnns(ann_ids)
        img_bgr = cv2.imread(str(sharp_path))
        if img_bgr is None:
            val_stats["skipped"] += 1; continue
        h, w = img_bgr.shape[:2]
        lines = [coco_ann_to_yolo(a, w, h, cat_to_idx)
                 for a in anns if a["bbox"][2] > 2 and a["bbox"][3] > 2]
        if lines:
            write_yolo_label(DATASET_DIR/"labels"/"val"/f"{stem}.txt", lines)
            val_stats["labels"] += 1

    print(f"\nTrain — deblurred: {train_stats['deblurred']}  "
          f"labels: {train_stats['labels']}  skipped: {train_stats['skipped']}")
    print(f"Val   — copied: {val_stats['copied']}  "
          f"labels: {val_stats['labels']}  skipped: {val_stats['skipped']}")

    return train_stats, val_stats


# ── 5. Write data.yaml ────────────────────────────────────────

def write_data_yaml(idx_to_name):
    num_classes = len(idx_to_name)
    names       = [idx_to_name[i] for i in range(num_classes)]
    yaml_path   = DATASET_DIR / "data.yaml"
    with open(yaml_path, "w") as f:
        f.write(f"path: {DATASET_DIR.resolve()}\n")
        f.write(f"train: images/train\n")
        f.write(f"val:   images/val\n")
        f.write(f"nc: {num_classes}\n")
        f.write(f"names: {names}\n")
    print(f"Saved data.yaml → {yaml_path}")
    return yaml_path


# ── 6. Augmentation config ────────────────────────────────────
# Passed to YOLO training — augments with blur, flips, mosaic, HSV

AUG_ARGS = dict(
    mosaic    = 1.0,    # mosaic augmentation
    fliplr    = 0.5,    # horizontal flip
    degrees   = 5.0,    # rotation
    translate = 0.1,    # translation
    scale     = 0.5,    # scale jitter
    hsv_h     = 0.015,  # HSV hue
    hsv_s     = 0.7,    # HSV saturation
    hsv_v     = 0.4,    # HSV value
    # 'blur' removed to prevent SyntaxError
)


# ── 7. Train ──────────────────────────────────────────────────

def train_detector(yaml_path):
    run_dir = OUTPUT_DIR / "runs"
    model   = YOLO("yolov8m.pt")   # start from COCO pretrained weights

    print(f"\nFine-tuning YOLOv8-m for {EPOCHS} epochs…")
    results = model.train(
        data      = str(yaml_path),
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH,
        project   = str(run_dir),
        name      = "nafnet_deblur",
        exist_ok  = True,
        device    = 0 if torch.cuda.is_available() else "cpu",
        patience  = 10,       # early stopping
        save      = True,     # save checkpoints
        plots     = True,     # training plots
        verbose   = True,
        amp       = True,     # ← added
        freeze    = 10,       # ← freeze backbone, only train detection head
        lr0       = 0.001,    # ← lower LR to prevent forgetting
        **AUG_ARGS
    )
    return model, results


# ── 8. Evaluate: baseline vs fine-tuned ──────────────────────

def evaluate_models(baseline_path, finetuned_model, yaml_path):
    """
    Compare baseline pretrained YOLOv8 vs fine-tuned on val set.
    """
    print("\nEvaluating baseline (pretrained YOLOv8m)…")
    baseline = YOLO(baseline_path)
    base_res = baseline.val(data=str(yaml_path), imgsz=IMG_SIZE,
                             verbose=False, split="val")

    print("\nEvaluating fine-tuned model…")
    ft_res   = finetuned_model.val(data=str(yaml_path), imgsz=IMG_SIZE,
                                    verbose=False, split="val")

    print(f"\n{'='*55}")
    print(f"  Evaluation — Baseline vs Fine-tuned")
    print(f"{'='*55}")
    print(f"  {'Metric':<25}  {'Baseline':>10}  {'Fine-tuned':>10}")
    print(f"  {'-'*51}")
    print(f"  {'mAP@0.5':<25}  {base_res.box.map50:>10.4f}  {ft_res.box.map50:>10.4f}")
    print(f"  {'mAP@0.5:0.95':<25}  {base_res.box.map:>10.4f}  {ft_res.box.map:>10.4f}")
    print(f"  {'Precision':<25}  {base_res.box.mp:>10.4f}  {ft_res.box.mp:>10.4f}")
    print(f"  {'Recall':<25}  {base_res.box.mr:>10.4f}  {ft_res.box.mr:>10.4f}")
    print(f"{'='*55}")

    return base_res, ft_res


# ── 9. Plot training curves ───────────────────────────────────

def plot_training_curves():
    results_csv = OUTPUT_DIR / "runs" / "nafnet_deblur" / "results.csv"
    if not results_csv.exists():
        print("No results.csv found — skipping training curve plot.")
        return

    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Training Curves — NAFNet Deblur Fine-tune", fontsize=13, fontweight="bold")

    metrics = [
        ("train/box_loss", "val/box_loss", "Box Loss"),
        ("train/cls_loss", "val/cls_loss", "Class Loss"),
        ("metrics/mAP50",  "metrics/mAP50-95", "mAP"),
    ]
    for ax, (train_col, val_col, title) in zip(axes, metrics):
        if train_col in df.columns:
            ax.plot(df["epoch"], df[train_col], color="#6c88d4", label="Train")
        if val_col in df.columns:
            ax.plot(df["epoch"], df[val_col],   color="#e88b3a", label="Val")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend()
        ax.grid(linewidth=0.4, alpha=0.5)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved training curves.")


# ── 10. Main ──────────────────────────────────────────────────

def run_task4():
    # Load COCO annotations
    print("Loading COCO annotations…")
    coco        = COCO(str(COCO_ANNOT_FILE))
    cat_to_idx, idx_to_name = build_coco_yolo_maps(coco)
    print(f"COCO classes: {len(idx_to_name)}")

    # Load NAFNet
    print("\nLoading NAFNet…")
    nafnet = load_nafnet()

    # Build dataset
    print("\nBuilding dataset…")
    build_dataset(coco, cat_to_idx, nafnet,
                  blur_dir=COCO_BLUR_DIR,
                  sharp_dir=COCO_SHARP_DIR,
                  max_train=MAX_TRAIN,
                  max_val=MAX_VAL)

    # ← added: free NAFNet before training starts
    del nafnet
    torch.cuda.empty_cache()
    gc.collect()
    print("NAFNet freed from GPU.")

    # Write YOLO config
    yaml_path = write_data_yaml(idx_to_name)

    # Train
    finetuned_model, _ = train_detector(yaml_path)

    # Evaluate
    evaluate_models("yolov8m.pt", finetuned_model, yaml_path)

    # Plot curves
    plot_training_curves()

    # Save best checkpoint path
    best = OUTPUT_DIR / "runs" / "nafnet_deblur" / "weights" / "best.pt"
    print(f"\nBest checkpoint: {best}")
    print("Task 4 complete.")
    return finetuned_model


finetuned_model = run_task4()

Using device: cuda
Loading COCO annotations…
loading annotations into memory...
Done (t=0.82s)
creating index...
index created!
COCO classes: 80

Loading NAFNet…

Building dataset…
Total COCO val images: 5000
Matched 4952 images with blur + annotations
Train: 300 | Val: 50

Preparing training set (deblurring COCOBlur images)…


Train:  66%|██████▋   | 199/300 [01:48<00:51,  1.96it/s]

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 5.6it/s 0.7s0.3s
                   all         50       1012      0.494      0.589      0.577      0.358

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/20      2.69G      1.666      1.606      1.403        119        640: 100% ━━━━━━━━━━━━ 38/38 5.4it/s 7.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 5.6it/s 0.7s0.3s
                   all         50       1012      0.492      0.562      0.574      0.356

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/20      2.69G      1.612      1.556      1.383        189        640: 100% ━━━━━━━━━━━━ 38/38 5.4it/s 7.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 5.6it/s 0.7s0.3s
                   all         50    